In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..\..'))

from ai_service.app.services.SRP_Detection_Final           import get_srp_report
from ai_service.app.services.OCP_Detection_Final           import get_ocp_report
from ai_service.app.services.Liskov_Substitution_Principle import get_lsp_report
from ai_service.app.services.ISP_detect                    import get_isp_report
from ai_service.app.services.dependancy_principle          import get_dip_report

In [2]:
DATA_PATH = r'python-codes-labeled.xlsx'

df = pd.read_excel(DATA_PATH, dtype=str)
df = df.dropna()
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass
7,8,Python,class Shape:\n def area(self):\n pas...,Pass,Pass,Pass,Pass,Pass
8,9,Python,"from abc import ABC, abstractmethod\n\n\nclass...",Pass,Pass,Pass,Pass,Pass
9,10,Python,"from abc import ABC, abstractmethod\n\n\n# def...",Pass,Pass,Pass,Violation,Pass


In [3]:
def extract_status(result):
    # list of dictionaries
    if isinstance(result, list):
        return [item.get('status') for item in result]

    # single dictionary
    elif isinstance(result, dict):
        return result.get('status')

    # fallback
    return None

In [4]:
df['SRP_detections'] = df['code'].apply(
    lambda x: extract_status(get_srp_report(x))
)

df['OCP_detections'] = df['code'].apply(
    lambda x: extract_status(get_ocp_report(x))
)

df['LSP_detections'] = df['code'].apply(
    lambda x: extract_status(get_lsp_report(x))
)

df['ISP_detections'] = df['code'].apply(
    lambda x: extract_status(get_isp_report(x))
)

df['DIP_detections'] = df['code'].apply(
    lambda x: extract_status(get_dip_report(x))
)
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip,SRP_detections,OCP_detections,LSP_detections,ISP_detections,DIP_detections
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation,[Pass],Violation,Pass,Violation,Pass
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation,[Pass],Pass,Pass,Pass,Pass
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Review]",Pass,Pass,Pass,Violation
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation,"[Pass, Pass, Pass]",Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Violation
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation,"[Pass, Pass]",Pass,Pass,Pass,Pass
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass,"[Review, Pass, Pass]",Pass,Pass,Violation,Pass
7,8,Python,class Shape:\n def area(self):\n pas...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Violation
8,9,Python,"from abc import ABC, abstractmethod\n\n\nclass...",Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Pass
9,10,Python,"from abc import ABC, abstractmethod\n\n\n# def...",Pass,Pass,Pass,Violation,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Pass


In [5]:
def normalize_label(value):
    # Handle lists like ['Pass']
    if isinstance(value, list):
        value = value[0]
        
    value = str(value).lower()

    if "violation" in value:
        return "Violation"

    return "Pass"

In [6]:
from sklearn.metrics import accuracy_score

srp_accuracy = accuracy_score(
    df['srp'].apply(normalize_label),
    df['SRP_detections'].apply(normalize_label)
)

ocp_accuracy = accuracy_score(
    df['ocp'].apply(normalize_label),
    df['OCP_detections'].apply(normalize_label)
)

lsp_accuracy = accuracy_score(
    df['lsp'].apply(normalize_label),
    df['LSP_detections'].apply(normalize_label)
)

isp_accuracy = accuracy_score(
    df['isp'].apply(normalize_label),
    df['ISP_detections'].apply(normalize_label)
)

dip_accuracy = accuracy_score(
    df['dip'].apply(normalize_label),
    df['DIP_detections'].apply(normalize_label)
)

print("SRP Accuracy:", srp_accuracy)
print("OCP Accuracy:", ocp_accuracy)
print("LSP Accuracy:", lsp_accuracy)
print("ISP Accuracy:", isp_accuracy)
print("DIP Accuracy:", dip_accuracy)

SRP Accuracy: 0.65
OCP Accuracy: 0.42
LSP Accuracy: 0.6
ISP Accuracy: 0.73
DIP Accuracy: 0.44


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
print("SRP Classification Report:")
print(classification_report(df['srp'].apply(normalize_label), df['SRP_detections'].apply(normalize_label)))
y_true = df['srp'].apply(normalize_label)
y_pred = df['SRP_detections'].apply(normalize_label)

labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)

print("\nConfusion Matrix:")
print(cm_df)


SRP Classification Report:
              precision    recall  f1-score   support

        Pass       0.65      0.98      0.78        63
   Violation       0.75      0.08      0.15        37

    accuracy                           0.65       100
   macro avg       0.70      0.53      0.46       100
weighted avg       0.68      0.65      0.55       100


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   62                    1
Actual_Violation              34                    3


In [8]:
print("OCP Classification Report:")
print(classification_report(df['ocp'].apply(normalize_label), df['OCP_detections'].apply(normalize_label)))
y_true = df['ocp'].apply(normalize_label)
y_pred = df['OCP_detections'].apply(normalize_label)

labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)

print("\nConfusion Matrix:")
print(cm_df)

OCP Classification Report:
              precision    recall  f1-score   support

        Pass       0.41      0.98      0.58        41
   Violation       0.67      0.03      0.06        59

    accuracy                           0.42       100
   macro avg       0.54      0.50      0.32       100
weighted avg       0.56      0.42      0.28       100


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   40                    1
Actual_Violation              57                    2


In [9]:
print("LSP Classification Report:")
print(classification_report(df['lsp'].apply(normalize_label), df['LSP_detections'].apply(normalize_label)))
y_true = df['lsp'].apply(normalize_label)
y_pred = df['LSP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

LSP Classification Report:
              precision    recall  f1-score   support

        Pass       0.57      0.98      0.72        52
   Violation       0.90      0.19      0.31        48

    accuracy                           0.60       100
   macro avg       0.73      0.58      0.51       100
weighted avg       0.73      0.60      0.52       100


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   51                    1
Actual_Violation              39                    9


In [10]:
print("ISP Classification Report:")
print(classification_report(df['isp'].apply(normalize_label), df['ISP_detections'].apply(normalize_label)))
y_true = df['isp'].apply(normalize_label)
y_pred = df['ISP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

ISP Classification Report:
              precision    recall  f1-score   support

        Pass       0.76      0.92      0.83        71
   Violation       0.57      0.28      0.37        29

    accuracy                           0.73       100
   macro avg       0.66      0.60      0.60       100
weighted avg       0.70      0.73      0.70       100


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   65                    6
Actual_Violation              21                    8


In [11]:
print("DIP Classification Report:")
print(classification_report(df['dip'].apply(normalize_label), df['DIP_detections'].apply(normalize_label)))
y_true = df['dip'].apply(normalize_label)
y_pred = df['DIP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

DIP Classification Report:
              precision    recall  f1-score   support

        Pass       0.37      0.45      0.40        42
   Violation       0.52      0.43      0.47        58

    accuracy                           0.44       100
   macro avg       0.44      0.44      0.44       100
weighted avg       0.46      0.44      0.44       100


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   19                   23
Actual_Violation              33                   25


In [12]:
all_actual = []
all_predicted = []

principles = ['SRP', 'OCP', 'LSP', 'ISP', 'DIP']

for p in principles:
    all_actual.extend(df[p.lower()].apply(normalize_label))
    all_predicted.extend(df[f'{p}_detections'].apply(normalize_label))

overall_accuracy = accuracy_score(all_actual, all_predicted)

print("Overall Accuracy:", overall_accuracy)

Overall Accuracy: 0.568
